In [ ]:
import numpy as np
import geopandas as gpd
import shapely.geometry

grid_size = 39

mun = gpd.read_file("./output/maceio/osm_maceio.geojson")
mun = mun.to_crs("EPSG:4326")
gdf = mun[mun["name"] == "Maceió"].copy()
city_boundary = gdf.union_all()

bbox = gdf.total_bounds
min_lon, min_lat, max_lon, max_lat = bbox[2], bbox[1], bbox[0], bbox[3]

lon = np.linspace(min_lon, max_lon, grid_size + 1)
lat = np.linspace(min_lat, max_lat, grid_size + 1)

mask = np.zeros((grid_size, grid_size), dtype=np.float32)

for i in range(grid_size):
    for j in range(grid_size):
        cell_box = shapely.geometry.box(lon[j], lat[i], lon[j + 1], lat[i + 1])
        if city_boundary.intersects(cell_box):
            mask[i, j] = 1

np.save("./output/maceio/maceio_mask_final.npy", mask)
print(f"Mask shape: {mask.shape}, valid cells: {int(mask.sum())}/{grid_size**2}")